# Task 2: 主表-only 干净评估（对齐历史 0.641）

**目标**: 去掉 additional_benign 负样本，只用主表数据评估，得到和早期 Phase 3（AlphaMissense 0.641）可比的干净指标。

**背景**: 主表有 2089 行(含 222 正例)，additional_benign 90 行(全部负例)。

In [1]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"

# ===== 加载数据 =====
df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")
df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")

# 对齐校验
assert len(df_main) == len(df_feat) == 2179
assert (df_main["Gene"].values == df_feat["Gene"].values).all()

# 把 source / AlphaMissense 从主表合并到特征表
df_feat["source"] = df_main["source"].values
df_feat["AlphaMissense_score"] = df_main["AlphaMissense score"].values

# 特征列定义
ID_COLS = ["KEY", "Gene", "reloc_v3"]
META_COLS = ["source", "AlphaMissense_score"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + META_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]

X_full = df_feat[feat_cols].values.astype(np.float32)
y_5class = df_feat["reloc_v3"].values.astype(int)
y_bin = (y_5class > 0).astype(int)
groups = df_feat["Gene"].values
source_arr = df_feat["source"].values
am_score_arr = df_feat["AlphaMissense_score"].values.astype(np.float64)

n_total = len(y_bin)
n_pos = int(y_bin.sum())
n_neg = n_total - n_pos
base_rate = n_pos / n_total

print(f"数据加载完毕: n={n_total}, 正例={n_pos}, 负例={n_neg}, base_rate={base_rate:.4f}")
print(f"source 分布: {dict(zip(*np.unique(source_arr, return_counts=True)))}")
print(f"特征列数: {len(feat_cols)}")

# 统一 5 折 CV
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
print("统一 CV 已初始化 (StratifiedGroupKFold, n_splits=5, groups=Gene)")

# ===== CV 工具函数 =====
def cv_evaluate_binary(X, y, groups, sample_weight_mode="balanced"):
    """手动 StratifiedGroupKFold，每折内 Imputer→Scaler→XGBoost"""
    xgb_params = dict(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.5,
        objective="binary:logistic", eval_metric="aucpr",
        random_state=42, n_jobs=-1, tree_method="hist",
    )
    oof = np.zeros(len(y), dtype=np.float32)
    per_fold = []
    for fold, (tr_idx, te_idx) in enumerate(cv.split(X, y, groups)):
        X_tr_raw, X_te_raw = X[tr_idx], X[te_idx]
        y_tr = y[tr_idx]
        imp = SimpleImputer(strategy="median")
        scl = StandardScaler()
        X_tr = scl.fit_transform(imp.fit_transform(X_tr_raw)).astype(np.float32)
        X_te = scl.transform(imp.transform(X_te_raw)).astype(np.float32)
        sw = compute_sample_weight("balanced", y_tr)
        model = XGBClassifier(**xgb_params)
        model.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
        oof[te_idx] = model.predict_proba(X_te)[:, 1]
        y_te = y[te_idx]
        per_fold.append({
            "fold": fold,
            "auroc": roc_auc_score(y_te, oof[te_idx]),
            "auprc": average_precision_score(y_te, oof[te_idx]),
            "n": len(y_te), "n_pos": int(y_te.sum())
        })
    return oof, per_fold

def print_metrics(label, y_true, oof, per_fold=None):
    auroc = roc_auc_score(y_true, oof)
    auprc = average_precision_score(y_true, oof)
    n = len(y_true); n_pos = int(y_true.sum()); br = n_pos / n
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  n={n}, pos={n_pos}, base_rate={br:.4f}")
    print(f"  AUROC = {auroc:.4f}")
    print(f"  AUPRC = {auprc:.4f}  (base_rate={br:.4f})")
    if per_fold:
        fa = [f["auroc"] for f in per_fold]
        fp = [f["auprc"] for f in per_fold]
        print(f"  Per-fold AUROC: {[f'{v:.3f}' for v in fa]}  "
              f"mean={np.mean(fa):.4f} ± {np.std(fa):.4f}")
        print(f"  Per-fold AUPRC: {[f'{v:.3f}' for v in fp]}  "
              f"mean={np.mean(fp):.4f} ± {np.std(fp):.4f}")
    return {"label": label, "n": n, "n_pos": n_pos, "base_rate": br,
            "auroc": auroc, "auprc": auprc}

print("CV 工具函数就绪")


数据加载完毕: n=2179, 正例=235, 负例=1944, base_rate=0.1078
source 分布: {'additional_benign': np.int64(90), 'main': np.int64(2089)}
特征列数: 1288
统一 CV 已初始化 (StratifiedGroupKFold, n_splits=5, groups=Gene)
CV 工具函数就绪


## 2a. 筛选主表数据并评估

In [2]:
# ===== 主表子集 =====
main_mask = source_arr == "main"
X_main = X_full[main_mask]
y_main = y_bin[main_mask]
g_main = groups[main_mask]

print(f"主表-only: n={len(y_main)}, 正例={int(y_main.sum())}, "
      f"base_rate={y_main.sum()/len(y_main):.4f}")

print("\n在主表子集上用统一 CV 训练 XGBoost ...")
oof_main, folds_main = cv_evaluate_binary(X_main, y_main, g_main)
r_main = print_metrics("v3 XGBoost (主表-only)", y_main, oof_main, folds_main)


主表-only: n=2089, 正例=222, base_rate=0.1063

在主表子集上用统一 CV 训练 XGBoost ...

  v3 XGBoost (主表-only)
  n=2089, pos=222, base_rate=0.1063
  AUROC = 0.5784
  AUPRC = 0.1495  (base_rate=0.1063)
  Per-fold AUROC: ['0.601', '0.603', '0.554', '0.562', '0.593']  mean=0.5826 ± 0.0207
  Per-fold AUPRC: ['0.169', '0.188', '0.129', '0.215', '0.158']  mean=0.1717 ± 0.0290


## 2b. 全量训练作为对比

In [3]:
print("在全量数据上用统一 CV 训练 XGBoost ...")
oof_full, folds_full = cv_evaluate_binary(X_full, y_bin, groups)
full_auroc = roc_auc_score(y_bin, oof_full)
full_auprc = average_precision_score(y_bin, oof_full)
r_full = print_metrics("v3 XGBoost (全量)", y_bin, oof_full, folds_full)


在全量数据上用统一 CV 训练 XGBoost ...

  v3 XGBoost (全量)
  n=2179, pos=222, base_rate=0.1019
  AUROC = 0.5603
  AUPRC = 0.1308  (base_rate=0.1019)
  Per-fold AUROC: ['0.594', '0.594', '0.409', '0.631', '0.564']  mean=0.5583 ± 0.0776
  Per-fold AUPRC: ['0.133', '0.164', '0.089', '0.164', '0.153']  mean=0.1405 ± 0.0282


## Task 2 汇总: 主表-only vs 全量

In [4]:
print(f"\n{'='*70}")
print(f"  TASK 2 对比: 主表-only vs 全量")
print(f"  {'指标':<15s} {'主表-only':>12s} {'全量':>12s} {'差异':>12s}")
print(f"  {'-'*51}")
print(f"  {'AUROC':<15s} {r_main['auroc']:>12.4f} {full_auroc:>12.4f} "
      f"{r_main['auroc']-full_auroc:>+12.4f}")
print(f"  {'AUPRC':<15s} {r_main['auprc']:>12.4f} {full_auprc:>12.4f} "
      f"{r_main['auprc']-full_auprc:>+12.4f}")
print(f"  {'n':<15s} {r_main['n']:>12d} {n_total:>12d}")
print(f"  {'正例':<15s} {r_main['n_pos']:>12d} {n_pos:>12d}")
print(f"  {'base_rate':<15s} {r_main['base_rate']:>12.4f} {base_rate:>12.4f}")
print(f"{'='*70}")

delta = r_main['auroc'] - full_auroc
if abs(delta) < 0.02:
    print(f"\n✓ 主表-only AUROC 与全量差异仅 {delta:+.4f}，"
          f"additional_benign 未造成明显指标虚高")
else:
    print(f"\n⚠ 主表-only 与全量差异 {delta:+.4f}，需注意 additional_benign 的影响")



  TASK 2 对比: 主表-only vs 全量
  指标                   主表-only           全量           差异
  ---------------------------------------------------
  AUROC                 0.5784       0.5603      +0.0181
  AUPRC                 0.1495       0.1308      +0.0187
  n                       2089         2179
  正例                       222          222
  base_rate             0.1063       0.1019

✓ 主表-only AUROC 与全量差异仅 +0.0181，additional_benign 未造成明显指标虚高
